In [1]:
import arxiv

In [2]:
import os

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\AI-Research-Experimentation-Agent'

In [7]:
import arxiv
import logging
import time

logger = logging.getLogger(__name__)

def fetch_arxiv_papers(query: str, max_results: int = 5, delay_seconds: float = 3.0) -> list[dict]:

    client = arxiv.Client(
        page_size=max_results,
        delay_seconds=delay_seconds,
        num_retries=5
    )

    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    papers = []
    
    try:
        for result in client.results(search):
            paper_data = {
                "paper_id": result.entry_id.split("/")[-1],  
                "title": result.title,
                "authors": [author.name for author in result.authors],
                "published_year": result.published.year,
                "pdf_url": result.pdf_url,
            }
            papers.append(paper_data)
            
    except arxiv.HTTPError as e:
        logger.error(f"arXiv HTTP error encountered: {e}")
        if e.status == 429:
            logger.warning("Rate limit hit! Waiting 10 seconds before continuing...")
            time.sleep(10)
    except Exception as e:
        logger.error(f"Unexpected error fetching papers from arXiv: {e}")
        
    return papers

In [8]:
fetch_arxiv_papers(query="retrieval augmented generation")

[{'paper_id': '2506.06962v3',
  'title': 'AR-RAG: Autoregressive Retrieval Augmentation for Image Generation',
  'authors': ['Jingyuan Qi', 'Zhiyang Xu', 'Qifan Wang', 'Lifu Huang'],
  'published_year': 2025,
  'pdf_url': 'https://arxiv.org/pdf/2506.06962v3'},
 {'paper_id': '2504.13684v1',
  'title': 'Intelligent Interaction Strategies for Context-Aware Cognitive Augmentation',
  'authors': [' Xiangrong',
   ' Zhu',
   'Yuan Xu',
   'Tianjian Liu',
   'Jingwei Sun',
   'Yu Zhang',
   'Xin Tong'],
  'published_year': 2025,
  'pdf_url': 'https://arxiv.org/pdf/2504.13684v1'},
 {'paper_id': '2504.17204v1',
  'title': 'Factually: Exploring Wearable Fact-Checking for Augmented Truth Discernment',
  'authors': ['Chitralekha Gupta',
   'Hanjun Wu',
   'Praveen Sasikumar',
   'Shreyas Sridhar',
   'Priambudi Bagaskara',
   'Suranga Nanayakkara'],
  'published_year': 2025,
  'pdf_url': 'https://arxiv.org/pdf/2504.17204v1'},
 {'paper_id': '2504.14689v1',
  'title': 'Designing AI Systems that Augm

In [ ]:
import os
import requests
import pymupdf  
import logging

logger = logging.getLogger(__name__)

def download_pdf(pdf_url: str, save_path: str) -> bool:
    """Downloads a PDF from a URL and saves it locally."""
    try:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        response = requests.get(pdf_url, timeout=15)
        response.raise_for_status()

        with open(save_path, "wb") as f:
            f.write(response.content)
        return True
    except Exception as e:
        logger.error(f"Failed to download PDF from {pdf_url}: {e}")
        return False

def parse_pdf_text(pdf_path: str) -> list[dict]:
    """
    Extracts structured text page-by-page from a local PDF.
    Returns a list of dicts: [{'page_num': 1, 'text': '...'}, ...]
    """
    pages_content = []
    try:
        doc = pymupdf.open(pdf_path)
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text = page.get_text("text")  # Extracts clean plain text
            
            # Basic cleanup: omit completely empty pages
            if text.strip():
                pages_content.append({
                    "page_num": page_num + 1,
                    "text": text.strip()
                })
        doc.close()
    except Exception as e:
        logger.error(f"Error parsing PDF at {pdf_path}: {e}")

    return pages_content

In [12]:
from src.REA.ingestion.arxiv_fetcher import fetch_arxiv_papers
from src.REA.ingestion.pdf_parser import download_pdf, parse_pdf_text

# 1. Fetch metadata
papers = fetch_arxiv_papers(query="retrieval augmented generation", max_results=5)

# 2. Process each paper
processed_papers = []

for paper in papers:
    pdf_filename = f"data/raw_pdfs/{paper['paper_id']}.pdf"
    
    print(f"Downloading {paper['title']}...")
    if download_pdf(paper['pdf_url'], pdf_filename):
        print(f"Parsing {pdf_filename}...")
        extracted_pages = parse_pdf_text(pdf_filename)
        
        paper['pages'] = extracted_pages
        paper['total_pages'] = len(extracted_pages)
        processed_papers.append(paper)

# Verify the output
print(f"Successfully processed {len(processed_papers)} papers.")
print(f"Sample page 1 preview:\n{processed_papers[0]['pages'][0]['text'][:300]}...")

Parsing data/raw_pdfs/2506.06962v3.pdf...
Parsing data/raw_pdfs/2504.13684v1.pdf...
Parsing data/raw_pdfs/2504.17204v1.pdf...
Parsing data/raw_pdfs/2504.14689v1.pdf...
Parsing data/raw_pdfs/2411.18583v1.pdf...
Successfully processed 5 papers.
Sample page 1 preview:
arXiv:2506.06962v3  [cs.CV]  14 Jun 2025
AR-RAG: Autoregressive Retrieval Augmentation for
Image Generation
Jingyuan Qi* 1
Zhiyang Xu* 1
Qifan Wang2
Lifu Huang3
1Virginia Tech
2Meta
3 UC Davis
jingyq1@vt.edu
(a) Vanilla Image Generation
Prompt
(c) Patch-based Autoregressive Retrieval Augmentation (O...


In [23]:
# Hints for building src/REA/chunking/recursive.py

from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_paper_pages(
    processed_paper: dict, 
    chunk_size: int = 800, 
    chunk_overlap: int = 150
) -> list[dict]:
    
    # 1. Instantiate the RecursiveCharacterTextSplitter with separators: ["\n\n", "\n", " ", ""]
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", " ", ""],
        chunk_size = chunk_size,
        chunk_overlap  = chunk_overlap)

    chunk_list = []

    paper_id = processed_paper["paper_id"]
    title = processed_paper.get("title", "")
    authors = processed_paper.get("authors", [])
    published_year = processed_paper.get("published_year", 0)
    

    # 2. Loop through each page in processed_paper["pages"]
    for page in processed_paper["pages"]:

        page_num = page["page_num"]

        text_chunk = splitter.split_text(page["text"])

        for c_idx, chunk_text in enumerate(text_chunk):
            chunk_dict = {
                "chunk_id": f"{paper_id}_p{page_num}_c{c_idx}",
                "paper_id": paper_id,
                "page_number": page_num,
                "text": chunk_text,
                "metadata": {
                    "title": title,
                    "authors": authors,
                    "published_year": published_year
                }
            }

            chunk_list.append(chunk_dict)
        
    print(len(chunk_list))
    print(chunk_list[0])

    return chunk_list
    

    # 3. Split the page["text"] into text chunks
    # 4. For each split text chunk, construct the dictionary matching our Chunk Object Specification above.
    # 5. Return the full list of chunk objects for the paper.

In [22]:
from src.REA.ingestion.arxiv_fetcher import fetch_arxiv_papers
from src.REA.ingestion.pdf_parser import download_pdf, parse_pdf_text
from src.REA.chunking.recursive import chunk_paper_pages

# 1. Fetch & parse 1 paper
papers = fetch_arxiv_papers(query="retrieval augmented generation", max_results=1)
paper = papers[0]

pdf_path = f"data/raw_pdfs/{paper['paper_id']}.pdf"
download_pdf(paper['pdf_url'], pdf_path)
paper['pages'] = parse_pdf_text(pdf_path)

# 2. Run chunker on the actual paper
chunks = chunk_paper_pages(paper, chunk_size=800, chunk_overlap=150)

# 3. Inspect results
print(f"Paper Title: {paper['title']}")
print(f"Total Pages: {len(paper['pages'])}")
print(f"Total Chunks Created: {len(chunks)}\n")

print("--- SAMPLE CHUNK 0 ---")
print(f"ID: {chunks[0]['chunk_id']}")
print(f"Page: {chunks[0]['page_number']}")
print(f"Text snippet: {chunks[0]['text'][:200]}...")

Paper Title: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
Total Pages: 18
Total Chunks Created: 108

--- SAMPLE CHUNK 0 ---
ID: 2506.06962v3_p1_c0
Page: 1
Text snippet: arXiv:2506.06962v3  [cs.CV]  14 Jun 2025
AR-RAG: Autoregressive Retrieval Augmentation for
Image Generation
Jingyuan Qi* 1
Zhiyang Xu* 1
Qifan Wang2
Lifu Huang3
1Virginia Tech
2Meta
3 UC Davis
jingyq1...


In [19]:
import os
import sys

project_path = "D:\AI-Research-Experimentation-Agent"

os.chdir(project_path)

# 2. Add it to the Python path just to be safe
if project_path not in sys.path:
    sys.path.append(project_path)

print("Current Working Directory updated to:", os.getcwd())

Current Working Directory updated to: D:\AI-Research-Experimentation-Agent


<>:4: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:4: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
C:\Users\DELL\AppData\Local\Temp\ipykernel_5968\832428339.py:4: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
  project_path = "D:\AI-Research-Experimentation-Agent"


In [24]:
%pwd

'D:\\AI-Research-Experimentation-Agent'

In [30]:
from src.REA.retrieval.vector_store import VectorStoreManager

In [31]:
store = VectorStoreManager()
store.add_chunks(chunks)  # Pass the 108 chunks you generated!

# Run a test query
results = store.query("How does autoregressive retrieval work in image generation?", top_k=2)
print("Top Result:", results['documents'][0][0][:200])
print("Metadata:", results['metadatas'][0][0])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3155.76it/s]


Top Result: In this paper, we propose Autoregressive Retrieval Augmentation (AR-RAG), a novel retrieval-
augmented paradigm for image generation that dynamically and autoregressively incorporates patch-
level k-n
Metadata: {'page_number': 2, 'title': 'AR-RAG: Autoregressive Retrieval Augmentation for Image Generation', 'published_year': 2025, 'paper_id': '2506.06962v3'}


In [32]:
from rank_bm25 import BM25Okapi
from typing import List, Dict, Any
from src.REA.retrieval.vector_store import VectorStoreManager

class HybridRetriever:
    def __init__(self, vector_store: VectorStoreManager):
        self.vector_store = vector_store
        self.bm25 = None
        self.corpus_chunks = []

    def index_chunks(self, chunks: List[Dict[str, Any]]):
        """Indexes chunks into both ChromaDB and BM25."""
        self.corpus_chunks = chunks
        
        # 1. Index into Vector Store
        self.vector_store.add_chunks(chunks)
        
        # 2. Tokenize text for BM25 (simple space/lowercase split)
        tokenized_corpus = [c["text"].lower().split(" ") for c in chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def _reciprocal_rank_fusion(
        self, 
        vector_results: List[str], 
        bm25_results: List[str], 
        k: int = 60
    ) -> List[str]:
        """Calculates RRF score for documents across both rank lists."""
        scores = {}

        # Process Vector search rankings
        for rank, chunk_id in enumerate(vector_results):
            if chunk_id not in scores:
                scores[chunk_id] = 0.0
            scores[chunk_id] += 1.0 / (k + rank + 1)

        # Process BM25 rankings
        for rank, chunk_id in enumerate(bm25_results):
            if chunk_id not in scores:
                scores[chunk_id] = 0.0
            scores[chunk_id] += 1.0 / (k + rank + 1)

        # Sort chunk IDs by highest combined RRF score
        sorted_chunks = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [chunk_id for chunk_id, score in sorted_chunks]

    def search(self, query_text: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """Executes Hybrid Search combining Dense Vector and Sparse BM25."""
        # 1. Vector Search
        v_raw = self.vector_store.query(query_text, top_k=top_k * 2)
        vector_ids = v_raw["ids"][0] if v_raw["ids"] else []

        # 2. BM25 Search
        tokenized_query = query_text.lower().split(" ")
        bm25_top_indices = self.bm25.get_top_n(tokenized_query, range(len(self.corpus_chunks)), n=top_k * 2)
        bm25_ids = [self.corpus_chunks[idx]["chunk_id"] for idx in bm25_top_indices]

        # 3. Fuse Rankings with RRF
        fused_ids = self._reciprocal_rank_fusion(vector_ids, bm25_ids)[:top_k]

        # Return full chunk objects matching the top fused IDs
        chunk_map = {c["chunk_id"]: c for c in self.corpus_chunks}
        return [chunk_map[cid] for cid in fused_ids if cid in chunk_map]

In [34]:
retriever = HybridRetriever(store)
retriever.index_chunks(chunks=chunks)
retriever.search("How does autoregressive retrieval work in image generation?")

[{'chunk_id': '2506.06962v3_p9_c5',
  'paper_id': '2506.06962v3',
  'page_number': 9,
  'text': 'These methods enable context-aware and prompt-sensitive guidance during generation. Another line\nof work [46] encodes multimodal retrievals into discrete visual and text tokens, and uses them directly\nas contextual input to augment the generation process of a multimodal large language model. All of\nthese works differe from our method by that our method works on patch-level, enabling more fine\ngrain retrievals and can dynamically adjust retrievals based on evolving generation states.\n7\nConclusion\nIn this work, we propose Autoregressive Retrieval Augmentation (AR-RAG), a novel retrieval\nparadigm that enhances image synthesis by leveraging k-nearest neighbor retrievals at the patch level.\nUnlike traditional image-level retrieval approaches, AR-RAG enables fine-grained visual element',
  'metadata': {'title': 'AR-RAG: Autoregressive Retrieval Augmentation for Image Generation',
   'aut

In [11]:
import os
api_key = os.environ.get("GCP_API_KEY")


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI


class QueryExpander:

    def __init__(self, llm=None):

        self.llm = llm or ChatGoogleGenerativeAI(
            model="gemini-3.7-flash",
            temperature=0.3
        )

        self.prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                (
                    "You are an AI research assistant. Given a user query "
                    "about an academic paper, generate 3 distinct search "
                    "query variations to retrieve relevant context:\n"
                    "1. A version using technical academic keywords.\n"
                    "2. A version focusing on underlying concepts or methodologies.\n"
                    "3. A version focusing on metrics, evaluation, or baseline comparisons.\n\n"
                    "Return ONLY the 3 variations separated by newlines, "
                    "with no numbering, bullets, or extra text."
                )
            ),
            ("human", "{query}")
        ])

    def expand_query(self, original_query: str) -> list[str]:

        chain = self.prompt | self.llm

        response = chain.invoke({
            "query": original_query
        })

        if isinstance(response.content, str):
            text = response.content

        else:
            text = "\n".join(
                block["text"]
                for block in response.content
                if isinstance(block, dict)
                and block.get("type") == "text"
            )

        raw_lines = text.strip().splitlines()

        queries = [
            line.strip()
            for line in raw_lines
            if line.strip()
        ]

        # Keep the original query
        if original_query not in queries:
            queries.insert(0, original_query)

        return queries

In [9]:
expander = QueryExpander()
queries = expander.expand_query("How does AR-RAG work?")

for i, q in enumerate(queries):
    print(f"Query {i}: {q}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Query 0: How does AR-RAG work?
Query 1: "AR-RAG" "autoregressive retrieval-augmented generation" architecture pipeline
Query 2: autoregressive RAG iterative dynamic query reformulation document retrieval methodology
Query 3: AR-RAG evaluation performance benchmark comparison baseline RAG metrics accuracy latency
